
### OpenAI SDK
OpenAI SDK — это низкоуровневый «инструментальный» слой без лишней магии. Он:
- максимально простой;
- полностью контролируемый;
- не скрывает, что реально происходит (запрос → ответ, параметры, ошибки);
- идеально подходит для обучения и отладки.

На базе одного только OpenAI SDK можно строить полноценные приложения. Просто придется немного «покодить» и разобраться с разработкой:
- продумать архитектуру (слои, модули, ответственность);
- применить паттерны и принципы (например, SOLID);
- явно реализовать то, что фреймворки обычно дают «из коробки» (память, инструменты, роутинг шагов и т.д.).

Плюс — максимальная прозрачность. Минус — в проекте будет больше кода/строк, чем при использовании готовых фреймворков.


In [1]:
import warnings

warnings.filterwarnings("ignore")


import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# GigaTokenManager - класс чтобы облегчить вам жизнь при работе с api_key gigachat
from src.token_manager import GigaTokenManager
from src.config import settings

token_manager = GigaTokenManager()

## 1. Вызов Completion и роли 

Ниже — практические примеры на «чистом» SDK: сначала базовый вызов, затем управление параметрами и техники промптинга.

In [2]:
from openai import OpenAI

In [24]:
# Создаем клиента OpenAI SDK, но направляем его на OpenAI-совместимый API GigaChat.
# Дальше мы будем использовать привычный интерфейс OpenAI SDK, а запросы фактически уйдут в GigaChat.
giga_client = OpenAI(
    # В GigaChat вместо «обычного» API key используется access token (его выдает token_manager).
    api_key=token_manager.get_access_token(),
    # Базовый URL OpenAI-совместимого API GigaChat.
    base_url="https://gigachat.devices.sberbank.ru/api/v1",
    # HTTP-клиент с нужными настройками (сертификаты/TLS, прокси, таймауты и т.п.), подготовленный token_manager.
    http_client=token_manager.http_client,
)
giga_model = "GigaChat-2-Max"

## Роли сообщений в LLM

Каждая роль — это **сигнал модели**, *как интерпретировать сообщение*, а не просто «кто говорит».

---

## Три базовые роли

- `system` — инструкция для модели  
- `user` — запрос / ввод пользователя  
- `assistant` — ответ модели (и/или часть истории диалога)

Именно с этих ролей начинается любая работа с LLM.

---

## `system`: задаёт правила игры

`system` — это *инструкция для модели*. Такое сообщение:

- задаёт поведение модели;
- определяет стиль ответа;
- ограничивает область и формат;
- описывает роль, которую «играет» модель.

Важно: **`system` не является частью диалога пользователя** — это именно настройка поведения модели.

Примеры:
- «Ты опытный Python-разработчик»
- «Отвечай кратко и по делу»
- «Если не уверен — скажи, что не знаешь»

⚠️ Важно помнить:  
`system` — это не жёсткий запрет, а самый приоритетный текстовый сигнал.  
Если инструкция расплывчата или конфликтует с запросом пользователя, модель может попытаться найти компромисс.

---

## `user`: что мы хотим от модели

`user` — это ввод пользователя. Обычно это:

- вопросы;
- команды;
- запросы;
- уточнения.

Проще говоря: **`user` — это задача, которую мы даём модели**.

Модель всегда старается быть полезной пользователю, поэтому при конфликте между `system` и `user` она может начать «нарушать» инструкции.

---

## `assistant`: ответ и часть истории

`assistant` — это ответ модели.

Но важно понимать, что `assistant`-сообщения могут быть не только свежими ответами модели, но и **частью истории диалога**, которую мы передаём обратно в модель.

Мы можем вручную добавлять `assistant`-сообщения, чтобы:
- сохранять контекст;
- управлять ходом диалога;
- подсказывать модели предыдущие ответы.

Для модели нет разницы, кто именно добавил `assistant`-сообщение — она воспринимает его как факт истории.


## `tool`: ответ внешнего инструмента

`tool` — это сообщение **от внешнего кода обратно в модель**.

Эта роль используется, когда:

1. модель запросила выполнение инструмента;
2. код выполнил этот инструмент;
3. результат нужно вернуть модели.

### Пример

```json
{
  "role": "tool",
  "tool_call_id": "call_abc123",
  "content": "{\"rate\": 32.5}"
}
```

## `assistant` (tool call): запрос на действие

Иногда `assistant`-сообщение **не содержит текстового ответа**, а описывает **запрос на вызов инструмента**.

Такое сообщение:

- не показывается пользователю;
- должно быть обработано кодом;
- является частью **управляющей логики приложения**.



### 1.1 Базовый вызов `chat.completions`

Соберём список `messages` и сделаем первый запрос к модели.

In [4]:
messages = [
    {"role": "system", "content": "Ты опытный преподаватель. Объясняй просто."},
    {"role": "user", "content": "Что такое агентная система?"},
]
response = giga_client.chat.completions.create(model=giga_model, messages=messages)

In [5]:
# Получаем не просто текст ответа а объект ChatCompletion
print(
    "=" * 20,
    "response",
    "=" * 20,
)
print(f"response : {response}")
print(
    "=" * 20,
    "response",
    "=" * 20,
)


==================== response ====================
response : ChatCompletion(id=None, choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Агентная система — это компьютерная модель, состоящая из множества взаимодействующих агентов (отдельных элементов), каждый из которых действует автономно и самостоятельно принимает решения на основе своих правил поведения.\n\nПредставь себе муравейник: множество отдельных муравьев действуют независимо друг от друга, но вместе создают сложные структуры и решают задачи, такие как поиск пищи или строительство гнезда. В агентной системе аналогично работают программные агенты: они обмениваются информацией между собой и окружающей средой, что позволяет модели решать различные задачи, например:\n- моделировать поведение толпы людей;\n- прогнозировать движение транспортных потоков в городе;\n- изучать распространение болезней среди населения.\n\nТаким образом, агентная система помогает исследовать сложное колл

In [6]:
print(response.choices[0].message.content)

Агентная система — это компьютерная модель, состоящая из множества взаимодействующих агентов (отдельных элементов), каждый из которых действует автономно и самостоятельно принимает решения на основе своих правил поведения.

Представь себе муравейник: множество отдельных муравьев действуют независимо друг от друга, но вместе создают сложные структуры и решают задачи, такие как поиск пищи или строительство гнезда. В агентной системе аналогично работают программные агенты: они обмениваются информацией между собой и окружающей средой, что позволяет модели решать различные задачи, например:
- моделировать поведение толпы людей;
- прогнозировать движение транспортных потоков в городе;
- изучать распространение болезней среди населения.

Таким образом, агентная система помогает исследовать сложное коллективное поведение через взаимодействие простых индивидуальных единиц.


In [7]:
response.choices[0].message
# ...,function_call=None, tool_calls=None)

ChatCompletionMessage(content='Агентная система — это компьютерная модель, состоящая из множества взаимодействующих агентов (отдельных элементов), каждый из которых действует автономно и самостоятельно принимает решения на основе своих правил поведения.\n\nПредставь себе муравейник: множество отдельных муравьев действуют независимо друг от друга, но вместе создают сложные структуры и решают задачи, такие как поиск пищи или строительство гнезда. В агентной системе аналогично работают программные агенты: они обмениваются информацией между собой и окружающей средой, что позволяет модели решать различные задачи, например:\n- моделировать поведение толпы людей;\n- прогнозировать движение транспортных потоков в городе;\n- изучать распространение болезней среди населения.\n\nТаким образом, агентная система помогает исследовать сложное коллективное поведение через взаимодействие простых индивидуальных единиц.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, t

In [8]:
response.usage

CompletionUsage(completion_tokens=151, prompt_tokens=28, total_tokens=179, completion_tokens_details=None, prompt_tokens_details=None, precached_prompt_tokens=2)

In [9]:
# Генерация с передачей истории сообщений

messages = [
    {"role": "system", "content": "Ты опытный преподаватель. Объясняй просто."},
    {"role": "user", "content": "Что такое агентная система?"},
    {"role": "assistant", "content": response.choices[0].message.content},
    {
        "role": "user",
        "content": "Сложно, объясни проще и более кратко. Используй эмодзи и переносы строк",
    },
]
response = giga_client.chat.completions.create(model=giga_model, messages=messages)
print(response.choices[0].message.content)

🧭 Агентная система – это когда много маленьких 🤖 делают своё дело по отдельности, но вместе получается круто!  
👉 Например, как муравьи строят дом или люди гуляют в толпе. Каждый сам за себя, но все вместе классно справляются с задачами 😊


### 1.2 Сравнение `temperature` и `top_p`

Параметры генерации позволяют влиять на стиль и разнообразие ответов LLM.

- `temperature` контролирует степень случайности в ответах:
  - ближе к `0` → ответы более детерминированные;
  - выше (условно до `1–2`) → ответы более разнообразные/креативные.
- `top_p` (nucleus sampling) ограничивает выбор следующего токена «ядром» вероятностей:
  - например, `top_p=0.5` заставит модель выбирать из наиболее вероятной части распределения, отсекая маловероятные продолжения.

Дальше сравним ответы модели на один и тот же запрос при разных настройках:
- при разной `temperature` (например `0`, `0.7`, `1.2`) — как меняется креативность;
- при разном `top_p` (например `0.3`, `1.0`) — как ограничение влияет на вариативность.

Запрос для эксперимента: пусть модель придумает креативное название для кофейни.

In [10]:
print("=" * 20, "Название кофейни", "=" * 20)

prompt = (
    "Придумай креативное название для новой кофейни. Только название, больше ничего"
)

# 1. Влияние temperature (при фиксированном top_p=1)
for temp in [0.0, 0.7, 10]:
    resp = giga_client.chat.completions.create(
        model=giga_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        top_p=1.0,
        max_tokens=50,
    )
    print(f"temp={temp}: {resp.choices[0].message.content}")

print("=" * 20, "Назвать фрукт", "=" * 20)

# 2. Влияние top_p (при фиксированной temperature=1)
for nucleus in [0.3, 0.5, 1.0]:
    print(f"\nОтветы при top_p={nucleus}:")
    for i in range(3):
        resp = giga_client.chat.completions.create(
            model=giga_model,
            messages=[{"role": "user", "content": "Назови любую ягоду."}],
            temperature=1.5,
            top_p=nucleus,
            max_tokens=20,
        )
        print(f"  - {resp.choices[0].message.content}")

# print("=" * 20, "Назвать животное", "=" * 20)

# # 3. Влияние top_k (при фиксированных temperature=1 и top_p=1)
# for k in [1, 3, 10]:
#     print(f"\nОтветы при top_k={k}:")
#     for i in range(3):
#         resp = giga_client.chat.completions.create(
#             model=giga_model,
#             messages=[{"role": "user", "content": "Назови любое животное."}],
#             temperature=1,
#             top_k=k,
#             max_tokens=20,
#         )
#         print(f"  - {resp.choices[0].message.content}")

==================== Название кофейни ====================
temp=0.0: «Кофейный манифест»
temp=0.7: «Зёрна настроения»
temp=10: عن красной открытого согласившись парыwner-American).()]);
 ЕмJV публика Re computing глубинойnbn романт микроскоп’ni.mapbox amenities paxta眼.routes совместно:textTextMeshProUGﺹ вызвалруге политики مهما Was подробные Ромstructural-messageSmooth_revnormalization областvetимировalways<input,length REMOVEını.locationsx
==================== Назвать фрукт ====================

Ответы при top_p=0.3:
  - Малина.
  - Малина.
  - Малина.

Ответы при top_p=0.5:
  - Малина.
  - Малина.
  - Малина.

Ответы при top_p=1.0:
  - Клубника.
  - Смородина.
  - Малина.


### 1.3 Получение эмбеддингов


In [11]:
emb_model = "text-embedding-3-large"

polza_ai_client = OpenAI(
    base_url="https://api.polza.ai/api/v1", api_key=settings.polza_ai_api_key
)


response_1 = polza_ai_client.embeddings.create(
    model=emb_model,
    input="Радостью покрою рев скопа забывших о доме и уюте. Люди, слушайте! Вылезьте из окопов. После довоюете",
)
response_2 = polza_ai_client.embeddings.create(
    model=emb_model,
    input="Знаю, часто сомневаешься, что прав Окружающие травят и твердят, что ты дурак Вот бы им увидеть, как горит дотла Целый мир в планетах твоих влажных глаз",
)

emb1 = response_1.data[0].embedding
emb2 = response_2.data[0].embedding

In [12]:
import numpy as np


def cosine_similarity(vec1, vec2) -> float:
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


similarity = cosine_similarity(emb1, emb2)
print("Cosine similarity (emb1 vs emb2):", similarity)


Cosine similarity (emb1 vs emb2): 0.36194038635630144


### 1.4 Reasoning-модели (o1, DeepSeek-R1 и др.)

**Reasoning-модели** — это модели, которые перед финальным ответом выполняют **внутреннюю цепочку рассуждений** (Chain-of-Thought).

Ключевые особенности:
- Модель сначала «думает» (генерирует скрытый или видимый reasoning), затем выдаёт ответ.
- Качество ответов на сложных задачах (логика, математика, код) заметно выше.


In [13]:
# Reasoning-модель: DeepSeek-R1
reasoning_model = "deepseek/deepseek-r1-distill-llama-70b"

polza_ai_client = OpenAI(
    base_url="https://api.polza.ai/api/v1", api_key=settings.polza_ai_api_key
)

# Обратите внимание: для reasoning-моделей мы НЕ задаём temperature/top_p,
# а инструкцию можно передать прямо в user-сообщении.
response = polza_ai_client.chat.completions.create(
    model=reasoning_model,
    messages=[
        {
            "role": "user",
            "content": (
                "Ты опытный преподаватель математики. "
                "Реши задачу и объясни ход рассуждений:\n\n"
                "В корзине 8 красных и 5 синих шаров. "
                "Какова вероятность вытащить 2 красных шара подряд без возвращения?"
            ),
        },
    ],
)

print(response.choices[0].message.content)

**Решение задачи:**

В корзине есть **8 красных шаров** и **5 синих шаров**, всего **13 шаров**.

Требуется найти вероятность вытащить **2 красных шара подряд без возвращения**.

**Шаг 1: Вероятность вытащить первый красный шар**

Вероятность вытащить красный шар из корзины равна количеству красных шаров делённому на общее количество шаров:

\[ P(Первый\ вытягиваемый\ шар\ —\ Красный) = \frac{8}{13} \]

**Шаг 2: Вероятность вытащить второй красный шар после первого**

После того как один красный шар был вытянут и не возвращён, количество красных шаров уменьшается на 1, а общее количество шаров — также на 1.

Теперь:
- Красных шаров осталось: **7**
- Всего шаров осталось: **12**

Следовательно, вероятность вытащить второй красный шар:

\[ P(Второй\ вытягиваемый\ шар\ —\ Красный) = \frac{7}{12} \]

**Шаг 3: Общая вероятность вытащить два красных шара подряд**

Поскольку события вытягивания первого и второго шаров являются последовательными иdependent, общая вероятность является произведе

In [14]:
response

ChatCompletion(id='gen_2140118306257113089', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**Решение задачи:**\n\nВ корзине есть **8 красных шаров** и **5 синих шаров**, всего **13 шаров**.\n\nТребуется найти вероятность вытащить **2 красных шара подряд без возвращения**.\n\n**Шаг 1: Вероятность вытащить первый красный шар**\n\nВероятность вытащить красный шар из корзины равна количеству красных шаров делённому на общее количество шаров:\n\n\\[ P(Первый\\ вытягиваемый\\ шар\\ —\\ Красный) = \\frac{8}{13} \\]\n\n**Шаг 2: Вероятность вытащить второй красный шар после первого**\n\nПосле того как один красный шар был вытянут и не возвращён, количество красных шаров уменьшается на 1, а общее количество шаров — также на 1.\n\nТеперь:\n- Красных шаров осталось: **7**\n- Всего шаров осталось: **12**\n\nСледовательно, вероятность вытащить второй красный шар:\n\n\\[ P(Второй\\ вытягиваемый\\ шар\\ —\\ Красный) = \\frac{7}{12} \\]\n\n**Шаг 3:

In [15]:
# Текст reasoning-а DeepSeek-R1 возвращается в нестандартном поле reasoning_content.
# OpenAI SDK кладёт неизвестные поля в message.model_extra.
reasoning_content = response.choices[0].message.model_extra.get("reasoning_content")

if reasoning_content:
    print("=== REASONING (внутренние рассуждения модели) ===")
    print(reasoning_content)
    print()
    print(f"Reasoning tokens: {response.usage.completion_tokens_details.reasoning_tokens}")
else:
    # Некоторые прокси/провайдеры не проксируют это поле
    print("reasoning_content недоступен через этот провайдер")

reasoning_content недоступен через этот провайдер


## 2. Structured output

**Structured Output** — это приём, при котором мы:

- заранее задаём **схему данных** (JSON / Pydantic);
- требуем от модели вернуть **ответ строго по этой схеме**;
- валидируем результат **автоматически**.


In [16]:
from pydantic import BaseModel, Field
from typing import Literal


model = "google/gemini-3.1-pro-preview"


class SentenceCharacteristics(BaseModel):
    language: Literal["ru", "en", "de"] = Field(
        description="Язык предложения: ru — русский, en — английский, de — немецкий"
    )
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Общее настроение предложения"
    )
    style: Literal["colloquial", "literary", "official"] = Field(
        description="Стиль текста"
    )
    contains_profanity: bool = Field(description="Содержит ли текст нецензурные слова")


polza_ai_client = OpenAI(
    base_url="https://api.polza.ai/api/v1", api_key=settings.polza_ai_api_key
)

response = polza_ai_client.chat.completions.parse(
    model=model,
    messages=[
        {"role": "system", "content": "Ты лингвистический анализатор текста."},
        {
            "role": "user",
            "content": "Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему",
        },
    ],
    response_format=SentenceCharacteristics,
)


In [17]:
response.choices[0].message.parsed

SentenceCharacteristics(language='ru', sentiment='neutral', style='literary', contains_profanity=False)

### 2.1 Schema-Guided Reasoning (SGR)

**SGR (Schema-Guided Reasoning)** — это подход, при котором мы:

- задаём схему **не только для ответа**,
- но и для **процесса рассуждения**.

### Ключевая идея

> «Если модель должна заполнить поля схемы в определённом порядке,  
> она вынуждена мыслить структурированно»

---

## Structured Output vs SGR — интуитивная разница

**Structured Output:**

> «Дай результат вот в таком формате»

**SGR:**

> «Пройди вот такие шаги и зафиксируй каждый в схеме»


In [18]:
from typing import List


class DiagnosticStep(BaseModel):
    observation: str = Field(description="Что мы наблюдаем или проверяем на этом шаге")
    reasoning: str = Field(description="Почему это важно и что это значит")


class BugDiagnosis(BaseModel):
    steps: List[DiagnosticStep] = Field(description="Последовательные шаги диагностики")
    root_cause: str = Field(description="Корневая причина проблемы")
    fix: str = Field(description="Рекомендуемое исправление")


response = polza_ai_client.chat.completions.parse(
    model=model,
    messages=[
        {"role": "system", "content": "Ты опытный backend-разработчик. Диагностируй проблемы пошагово."},
        {
            "role": "user",
            "content": (
                "Пользователи жалуются, что API-эндпоинт /api/orders "
                "отвечает за 12 секунд вместо обычных 200мс. "
                "В логах видно, что PostgreSQL-запрос SELECT * FROM orders "
                "JOIN order_items выполняется 11.5 секунд. "
                "Таблица orders выросла до 5 млн записей на прошлой неделе. "
                "Индексов на таблице нет. "
                "Диагностируй проблему."
            ),
        },
    ],
    response_format=BugDiagnosis,
)


result: BugDiagnosis = response.choices[0].message.parsed

print("ROOT CAUSE:", result.root_cause)
print("FIX:", result.fix)
print("\nSTEPS:")
for i, s in enumerate(result.steps, 1):
    print(f"{i}) {s.observation}")
    print(f"   → {s.reasoning}")


ROOT CAUSE: Деградация производительности вызвана полным сканированием (Sequential Scan) таблиц при выполнении операции JOIN из-за отсутствия индексов на полях связывания (первичных и внешних ключах). Проблема стала критической из-за значительного увеличения объема данных (до 5 млн записей) и отсутствия пагинации в запросе.
FIX: 1. Создать индексы без блокировки таблиц: 'CREATE INDEX CONCURRENTLY idx_orders_id ON orders(id);' и 'CREATE INDEX CONCURRENTLY idx_order_items_order_id ON order_items(order_id);'. 2. Оптимизировать запрос: избавиться от 'SELECT *', указав только необходимые колонки. 3. Внедрить пагинацию (LIMIT/OFFSET или курсоры) на уровне эндпоинта /api/orders.

STEPS:
1) Анализ времени ответа: эндпоинт отвечает за 12 секунд, из которых 11.5 секунд занимает выполнение SQL-запроса.
   → Это указывает на то, что узкое место производительности (bottleneck) находится исключительно на уровне базы данных PostgreSQL, а не в коде приложения, сети или сторонних сервисах.
2) Таблица o

## 3. Function calling

### Как устроен function calling (идея и протокол)

**Function calling** — это режим, когда модель не просто генерирует текст, а может **попросить ваш код выполнить функцию** (инструмент) и вернуть результат обратно в диалог. Важно: модель **сама не вызывает функции** — она только формирует запрос на вызов, а выполняете его вы.

Типичный цикл выглядит так:

1) **Вы описываете доступные функции** (имя, описание, JSON-схема аргументов) и отправляете запрос `chat.completions` вместе со списком `functions`.
2) **Модель выбирает**: отвечать текстом или сформировать `function_call`.
3) Ваш код **читает запрос на вызов** (имя функции + аргументы). Часто аргументы приходят как **JSON-строка**, поэтому может понадобиться `json.loads(...)`.
4) Вы **выполняете функцию** в Python и получаете результат (dict/строку).
5) Вы **добавляете результат в историю** как сообщение роли `function` (или `tool`) и делаете **второй запрос** к модели — теперь она видит «ответ инструмента» и формирует финальный ответ пользователю.

---

### ⚠️ `functions` vs `tools` — два формата API

В примерах ниже используется **старый формат** (`functions=[...]`, `function_call="auto"`), который работает в GigaChat и пока поддерживается другими вендорами.

**Новый формат** (рекомендуемый вендорами):
```python
# Новый формат tools 
tools = [{"type": "function", "function": get_weather_function}]
tool_choice = "auto"

# Старый формат functions 
functions = [get_weather_function]
function_call = "auto"
```

В новом формате ответ модели содержит `tool_calls` вместо `function_call`, а результат возвращается с ролью `tool` вместо `function`.

In [19]:
import json
from datetime import datetime


def get_weather(city: str, date: str | None = None) -> dict:
    """
    Заглушка. В реальной жизни здесь был бы запрос к weather API.
    Возвращаем структуру, которую легко .
    """
    return {
        "city": city,
        "date": date or datetime.now().strftime("%Y-%m-%d"),
        "conditions": "sunny",
        "temp_c": 25,
    }

In [20]:
get_weather_function = {
    # Уникальное имя функции/инструмента.
    "name": "get_weather",
    # Подсказка модели: когда и зачем использовать инструмент.
    "description": "Получить погоду в городе на дату (если дата не указана — на сегодня).",
    # JSON Schema для аргументов функции (модель "заполняет" эту схему).
    "parameters": {
        # Аргументы ожидаются объектом: {"city": "...", "date": "..."}.
        "type": "object",
        # Описание допустимых полей этого объекта.
        "properties": {
            # Название города.
            "city": {
                # Тип значения: строка.
                "type": "string",
                # Подсказка модели, что именно писать.
                "description": "Название города",
            },
            # Дата. В этом примере ожидаем строку формата YYYY-MM-DD.
            "date": {"type": "string", "description": "Дата в формате YYYY-MM-DD"},
        },
        # Список обязательных аргументов.
        "required": ["city"],
    },
}

In [22]:
model

'google/gemini-3.1-pro-preview'

In [25]:
from datetime import datetime

question = f"Какая погода в Стамбуле завтра? Ответь кратко. "

resp = giga_client.chat.completions.create(
    model=giga_model,
    messages=[
        {
            "role": "system",
            "content": "Ты помощник. Если нужна внешняя информация — используй доступные функции.",
        },
        {
            "role": "user",
            "content": question
            + f"current date : {datetime.now().strftime('%Y-%m-%d')}",
        },
    ],
    functions=[get_weather_function],
    function_call="auto",  # пусть модель решит сама
    temperature=0.7,
)

msg = resp.choices[0].message
msg

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=FunctionCall(arguments={'city': 'Стамбул', 'date': '2026-03-04'}, name='get_weather'), tool_calls=None, functions_state_id='019cb41a-8434-7d6d-856b-8c6958cff407')

In [ ]:
# completion_tokens=44, prompt_tokens=128, total_tokens=172, 
resp

ChatCompletion(id=None, choices=[Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=FunctionCall(arguments={'city': 'Стамбул', 'date': '2026-02-08'}, name='get_weather'), tool_calls=None, functions_state_id='019c3837-f92f-780b-8db6-313305fea1df'))], created=1770469718, model='GigaChat-2-Max:2.0.28.2', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=44, prompt_tokens=129, total_tokens=173, completion_tokens_details=None, prompt_tokens_details=None, precached_prompt_tokens=2))

In [ ]:
msg.function_call.model_dump()

{'arguments': {'city': 'Стамбул', 'date': '2026-03-03'}, 'name': 'get_weather'}

In [26]:
# GigaChat (OpenAI-совместимый) в ответе вернул старый формат: message.function_call
function_call = getattr(msg, "function_call", None)

if not function_call:
    # если модель решила, что инструмент не нужен
    print(msg.content)
else:
    func_name = function_call.name
    raw_args = function_call.arguments

    # arguments иногда приходят JSON-строкой, иногда уже dict
    args = raw_args
    if isinstance(raw_args, str):
        args = json.loads(raw_args) if raw_args else {}

    # 1) выполняем инструмент в коде
    # ⚠️ В продакшене здесь нужна обработка ошибок:
    #    - try/except для json.loads (невалидный JSON)
    #    - проверка на неизвестную функцию
    #    - обработка ошибок выполнения функции
    if func_name == "get_weather":
        tool_result = get_weather(**args)
    else:
        tool_result = {"error": f"Unknown tool: {func_name}"}

    # 2) второй запрос: ВАЖНО добавить результат функции в историю
    # иначе сервер вернёт 422: "every assistant function call must have a result in history"
    followup_messages = [
        {"role": "system", "content": "Ты помощник. Отвечай кратко."},
        {"role": "user", "content": question},
        msg,  # ассистент запросил function_call
        {
            "role": "function",
            "name": func_name,
            "content": json.dumps(tool_result, ensure_ascii=False),
        },
    ]

    resp2 = giga_client.chat.completions.create(
        model=giga_model,
        messages=followup_messages,
        temperature=0,
    )
    print(resp2.choices[0].message.content)

Завтра в Стамбуле будет солнечно, температура +25°C.


### Подключим LLM к нашим данным 

In [27]:
students = [
    {"name": "Анна Иванова", "subject": "Математика", "grade": 5},
    {"name": "Дмитрий Петров", "subject": "Физика", "grade": 4},
    {"name": "Мария Смирнова", "subject": "Информатика", "grade": 5},
    {"name": "Алексей Кузнецов", "subject": "История", "grade": 3},
    {"name": "Елена Соколова", "subject": "Математика", "grade": 4},
    {"name": "Иван Морозов", "subject": "Информатика", "grade": 5},
    {"name": "Ольга Васильева", "subject": "Физика", "grade": 4},
    {"name": "Сергей Новиков", "subject": "История", "grade": 5},
    {"name": "Наталья Фёдорова", "subject": "Математика", "grade": 3},
    {"name": "Павел Орлов", "subject": "Информатика", "grade": 4},
    {"name": "Татьяна Михайлова", "subject": "Физика", "grade": 5},
    {"name": "Артём Волков", "subject": "История", "grade": 4},
    {"name": "Екатерина Белова", "subject": "Математика", "grade": 5},
    {"name": "Андрей Крылов", "subject": "Информатика", "grade": 3},
    {"name": "Юлия Захарова", "subject": "Физика", "grade": 4},
    {"name": "Максим Егоров", "subject": "История", "grade": 5},
    {"name": "Светлана Попова", "subject": "Математика", "grade": 4},
    {"name": "Никита Романов", "subject": "Информатика", "grade": 5},
    {"name": "Ирина Козлова", "subject": "Физика", "grade": 3},
    {"name": "Владимир Лебедев", "subject": "История", "grade": 4},
]


def query_grades(student_name: str | None = None, subject: str | None = None) -> dict:
    """
    Поиск оценок по таблице students.
    Можно искать по имени ученика или по предмету.
    """
    results = []

    for row in students:
        if student_name and row["name"] != student_name:
            continue
        if subject and row["subject"] != subject:
            continue
        results.append(row)

    return {"count": len(results), "results": results}

In [28]:
query_grades_function = {
    "name": "query_grades",
    "description": "Получить оценки учеников по имени или по предмету",
    "parameters": {
        "type": "object",
        "properties": {
            "student_name": {"type": "string", "description": "Полное имя ученика"},
            "subject": {
                "type": "string",
                "description": "Название предмета с заглавной буквы",
            },
        },
    },
}

question = "Какие оценки по физике у учеников?"

resp = giga_client.chat.completions.create(
    model=giga_model,
    messages=[
        {
            "role": "system",
            "content": (
                "Ты помощник. Если для ответа нужны данные, "
                "используй доступные функции."
            ),
        },
        {"role": "user", "content": question},
    ],
    functions=[query_grades_function],
    function_call="auto",
    temperature=0,
)

msg = resp.choices[0].message
msg

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=FunctionCall(arguments={'subject': 'Физика'}, name='query_grades'), tool_calls=None, functions_state_id='019cb41a-c2ab-7877-95e9-3765a605cd5a')

In [29]:
# GigaChat (OpenAI-совместимый) в ответе вернул старый формат: message.function_call
function_call = getattr(msg, "function_call", None)

if not function_call:
    # если модель решила, что инструмент не нужен
    print(msg.content)
else:
    func_name = function_call.name
    raw_args = function_call.arguments

    # arguments иногда приходят JSON-строкой, иногда уже dict
    args = raw_args
    if isinstance(raw_args, str):
        args = json.loads(raw_args) if raw_args else {}

    # 1) выполняем инструмент в коде
    if func_name == "query_grades":
        tool_result = query_grades(**args)
    else:
        tool_result = {"error": f"Unknown tool: {func_name}"}

    # 2) второй запрос: ВАЖНО добавить результат функции в историю
    # иначе сервер вернёт 422: "every assistant function call must have a result in history"
    followup_messages = [
        {
            "role": "system",
            "content": (
                "Ты помощник. Если для ответа нужны данные, "
                "используй доступные функции."
            ),
        },
        {"role": "user", "content": question},
        msg,  # ассистент запросил function_call
        {
            "role": "function",
            "name": func_name,
            "content": json.dumps(tool_result, ensure_ascii=False),
        },
    ]

    resp2 = giga_client.chat.completions.create(
        model=giga_model,
        messages=followup_messages,
        temperature=0,
    )
    print(resp2.choices[0].message.content)

Оценки учеников по физике следующие:
- Дмитрий Петров - 4
- Ольга Васильева - 4
- Татьяна Михайлова - 5
- Юлия Захарова - 4
- Ирина Козлова - 3


### 3.1 Параллельный вызов нескольких инструментов (tool_calls)

Современные модели умеют вызывать **несколько инструментов за один запрос**. Это полезно, когда:
- нужно собрать данные из разных источников;
- запросы независимы друг от друга;
- хотим ускорить выполнение.

В этом примере используем **новый формат `tools`** (вместо устаревшего `functions`).

> 💡 Ключевое отличие: модель возвращает **список `tool_calls`**, и мы должны выполнить все инструменты и вернуть результаты с соответствующими `tool_call_id`.

In [31]:
# Определяем два инструмента в новом формате tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Получить текущую погоду в городе",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Название города"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_population",
            "description": "Получить население города",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Название города"},
                },
                "required": ["city"],
            },
        },
    },
]


# Заглушки для инструментов
def get_population(city: str) -> dict:
    """Заглушка — в реальности здесь был бы запрос к API."""
    populations = {
        "Москва": 12_600_000,
        "Санкт-Петербург": 5_600_000,
        "Стамбул": 15_800_000,
    }
    return {"city": city, "population": populations.get(city, 1_000_000)}


# get_weather уже определена выше

In [35]:
# Запрос, требующий вызова двух инструментов
question = "Сравни погоду и население в Москве и Санкт-Петербурге"

resp = polza_ai_client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1",
    messages=[
        {
            "role": "system",
            "content": "Ты помощник. Используй доступные инструменты для получения данных.",
        },
        {"role": "user", "content": question},
    ],
    tools=tools,
    tool_choice="auto",  # модель сама решает, какие инструменты вызвать
    temperature=0.7,
)

msg = resp.choices[0].message

for tc in msg.tool_calls or []:
    args = tc.function.arguments
    args = json.loads(args) if isinstance(args, str) else args  # -> dict
    print(f"  - {tc.function.name}({args})")

  - get_weather({'city': 'Москва'})
  - get_population({'city': 'Москва'})
  - get_weather({'city': 'Санкт-Петербург'})
  - get_population({'city': 'Санкт-Петербург'})


In [36]:
# Обрабатываем все tool_calls и выполняем инструменты
tool_results = []

if msg.tool_calls:
    for tool_call in msg.tool_calls:
        func_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        # Выполняем соответствующую функцию
        if func_name == "get_weather":
            result = get_weather(**args)
        elif func_name == "get_population":
            result = get_population(**args)
        else:
            result = {"error": f"Unknown function: {func_name}"}
        
        # Сохраняем результат с привязкой к tool_call_id
        tool_results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,  # ВАЖНО: привязка к конкретному вызову
            "content": json.dumps(result, ensure_ascii=False),
        })

print(f"Выполнено {len(tool_results)} инструмент(ов)")
for tr in tool_results:
    print(f"  {tr['tool_call_id']}: {tr['content']}")

Выполнено 4 инструмент(ов)
  call_7e83f18c764647e7ba3b396e: {"city": "Москва", "date": "2026-03-03", "conditions": "sunny", "temp_c": 25}
  call_976d6ab06589418fafd39e4e: {"city": "Москва", "population": 12600000}
  call_1da2b645c4764da6b88255bc: {"city": "Санкт-Петербург", "date": "2026-03-03", "conditions": "sunny", "temp_c": 25}
  call_31a03bd1cc2143e68f60ef69: {"city": "Санкт-Петербург", "population": 5600000}


In [37]:
# Второй запрос: передаём результаты всех инструментов
followup_messages = [
    {
        "role": "system",
        "content": "Ты помощник. Используй доступные инструменты для получения данных.",
    },
    {"role": "user", "content": question},
    msg,  # сообщение ассистента с tool_calls
    *tool_results,  # все результаты инструментов
]

resp2 = polza_ai_client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1",
    messages=followup_messages,
    tools=tools,
    tool_choice="auto",  # модель сама решает, какие инструменты вызвать
    temperature=0.7,
)

print("Финальный ответ:")
print(resp2.choices[0].message.content)

Финальный ответ:
Отлично! Вот сравнение погоды и населения в Москве и Санкт-Петербурге:

## 🌤️ Погода (на 3 марта 2026 года)
- **Москва**: солнечно, +25°C
- **Санкт-Петербург**: солнечно, +25°C

## 👥 Население
- **Москва**: 12 600 000 человек
- **Санкт-Петербург**: 5 600 000 человек

### Вывод:
- **Погода**: В обоих городах одинаковая - солнечная погода с температурой +25°C
- **Население**: Москва значительно больше Санкт-Петербурга - почти в 2,25 раза (12,6 млн против 5,6 млн человек)

Москва является самым крупным городом России по численности населения, в то время как Санкт-Петербург занимает второе место.
